# LLM prompting fundamentals: 

In the past few years language model have improved drastically. Indeed, language model can now write you syntactically perfect Python in the blink of and eye. That is no longer the difficult part (learning basic coding).

The hard part is that **the model cannot know about your problem context or your dataset.** 

It does not know the specific context of your project, the meaning of your data, or the constraints and limitations that shape your problem.

It will nevertheless produce confident, well-formatted, plausible-looking code built entirely on assumptions it invented based on what you provided.

**Core Idea : an LLM is highly capable but context dependent. These models are trained to always produce a plausible answer rather than admit uncertainty. So every fact you leave out of your prompt might lead the model to guess and provide the statistically most common fact from its training data. Sometimes that guess crashes. Other times it does not which is why it has to be used cautiously.**

### What you will do

You will prompt a model on the *same dataset* three times, changing **nothing but the prompt**.

| Round | Prompt | What comes back | How it fails |
|---|---|---|---|
| **1** | One lazy sentence | Code for a *different dataset* that the model invented | **Loudly**, it crashes or is out of context |
| **2** | Correct schema, nothing else | Careful code that passes every check you know | **Silently**, a plausible but wrong result |
| **3** | Full specification | Honest evaluation, an unflattering number, a defensible claim | It does not |


### How to use this notebook

Keep a chat window open next to it. Any model will do the trick (Claude, ChatGPT, Gemini). 

For each round:

1. Copy the suggested printed prompt
2. Run the prompt cell into your chat
3. Copy the code that comes back into the empty cell → **run it**

Your model will not answer exactly like mine. That is fine, and it is part of the lesson. Each round therefore gives you:

- a **"what to expect and why"** section *before* you run it. Read it first, then check whether your model behaved as predicted
- a **symptom checklist** to diagnose whatever your model actually produced
- a **reference cell** reproducing the typical answer, so the notebook works end-to-end even if your model surprises you

---
## Setup
Importing essential libraries

In [ ]:
try:
    import google.colab
    !pip install -q seaborn
except ImportError:
    pass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

# Display all DataFrame columns
pd.set_option("display.max_columns", None)
RANDOM_SEED = 42

---

## The dataset: Oxford Parkinson's Disease Telemonitoring

[**UCI Machine Learning Repository, dataset #189** — Tsanas & Little (2009).](https://archive.ics.uci.edu/dataset/189/parkinsons+telemonitoring)

The UPDRS (Unified Parkinson's Disease Rating Scale) is a measure of Parkinson's severity. Scoring it requires a neurologist, in a clinic, with the patient physically present. It is slow, expensive, and because it depends on a human rater it is subjective.

Parkinson's degrades speech very early: the voice becomes breathy, unstable in pitch and amplitude. 

A question arise : maybe a model could monitor disease progression based on cheap acoustic measurement taken from a microphone instead of relying on a specialist to do so.

**The study.** 42 people with early-stage Parkinson's were given a telemonitoring device at home. Each ran a recording **session** roughly weekly for six months. From each recording, 16 acoustic dysphonia measures were extracted :
- jitter = cycle-to-cycle variation in pitch
- shimmer = variation in amplitude
- NHR/HNR = noise-to-harmonics ratios
- RPDE, DFA, PPE = nonlinear signal-dynamics measures

## The task :

**Predict `total_UPDRS`** from the 16 voice measures using machine learning.


### Data inspection
Let's load the dataset first and inspect it to have a better first feeling of what it contains.

In [ ]:
from pathlib import Path


UCI_ID = 189
UCI_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "parkinsons/telemonitoring/parkinsons_updrs.data"
)
CACHE = Path("parkinsons_updrs.csv")

# Everything that is NOT a dysphonia measure: identifiers, covariates, targets.
NON_VOICE = ["subject#", "age", "sex", "test_time", "motor_UPDRS", "total_UPDRS"]


TARGET_NAME = "total_UPDRS"

def load_telemonitoring(cache: str | Path = CACHE) -> pd.DataFrame:
    """Load the UCI Parkinson's Telemonitoring dataset, with fallbacks.

    Tries, in order: a local cached CSV, the `ucimlrepo` package, then the
    UCI archive URL. Caches to disk so the notebook runs offline afterwards.
    """
    cache = Path(cache)
    if cache.exists():
        return pd.read_csv(cache)

    try:  # official UCI python package
        from ucimlrepo import fetch_ucirepo

        data = fetch_ucirepo(id=UCI_ID).data
        frame = (
            data.original
            if data.original is not None
            else pd.concat([data.features, data.targets], axis=1)
        )
        frame.to_csv(cache, index=False)
        return frame
    except Exception as err:
        print(f"ucimlrepo unavailable ({type(err).__name__}), trying direct download…")

    frame = pd.read_csv(UCI_URL)
    frame.to_csv(cache, index=False)
    return frame


def fetch_parkinson_telemonitoring(
    cache: str | Path = CACHE,
    drop_motor_updrs: bool = False,
) -> tuple[pd.DataFrame, dict]:
    """
    Load UCI #189 and describe its structure for `disguise_dataset`.

    Parameters
    ----------
    cache            : path of the cached CSV
    drop_motor_updrs : drop 'motor_UPDRS'.  It is a near-duplicate of the
                       target (r ≈ 0.95) and would hand the answer to any
                       model — a second, much cruder leak that would mask
                       the repeated-measures one we want to teach.

    Returns
    -------
    df   : the raw DataFrame (5875 rows × 21 columns, or 20 without motor)
    meta : dict with 'target', 'group_col', 'sequence_col', 'feature_cols',
           'categorical_cols', 'task', 'description'
    """
    df = load_telemonitoring(cache)

    if drop_motor_updrs and "motor_UPDRS" in df.columns:
        df = df.drop(columns=["motor_UPDRS"])

    # The 16 acoustic dysphonia measures: the only legitimate predictors.
    voice = [c for c in df.columns if c not in NON_VOICE]

    df["subject#"] = df["subject#"].astype(int)
    df["sex"] = df["sex"].astype(int)

    n_subjects = df["subject#"].nunique()
    reps = df.groupby("subject#").size()

    meta = {
        "target": "total_UPDRS",
        "group_col": "subject#",
        "sequence_col": "test_time",
        "feature_cols": voice + ["age", "sex"],
        "voice_cols": voice,
        "categorical_cols": ["sex"],
        "n_subjects": int(n_subjects),
        "description": (
            f"UCI #189: {n_subjects} patients, {int(reps.min())}-{int(reps.max())} "
            f"voice recordings each ({len(df)} rows). UPDRS moves slowly within a "
            f"patient, so a naive random split puts near-duplicate rows of the same "
            f"patient on both sides and leaks badly."
        ),
    }
    return df, meta

In [ ]:
# ── Fetch the REAL table: real column names, real values ──
df_raw, meta = fetch_parkinson_telemonitoring()

VOICE = meta["voice_cols"]
print("=" * 65)
print("  1. FETCHING RAW DATASET")
print("=" * 65)
print(f"  Shape       : {df_raw.shape}")
print(f"  Target      : {meta['target']}")
print(f"  Group col   : {meta['group_col']}")
print(f"  Subjects    : {meta['n_subjects']}")
print(f"  Voice cols  : {meta['voice_cols']}")
print(f"  Description : {meta['description']}")
print()
df_raw.head()


------
## A First Look at the Dataset Variables

Before we start coding, let's look at the features we will be analyzing. These variables measure different acoustic properties of voice recordings, which are often used in medical datasets (like tracking Parkinson's disease).

**Vocabulary (if these are new to you):**
* **Jitter** — How much the pitch wobbles from one vocal-fold cycle to the next.
* **Shimmer** — The same idea as Jitter, but for loudness (amplitude).
* **HNR / NHR** (Harmonics-to-Noise / Noise-to-Harmonics Ratio) — How much of the sound is a clean tone versus breathy noise. A high HNR means a clearer voice.
* **RPDE, DFA, PPE** — Nonlinear statistical summaries of how (ir)regular or unpredictable the signal is.


### Visualisation Exercise 1: Generate the Visualization Code using AI

What is now amazing with generative model is that they help us with boilerplate code (such as plotting) that we do over and over again when we do scientific coding.

Instead of writing the `matplotlib` code from scratch, let's use a Large Language Model (LLM) to generate a visualization for us.

**Instructions:**
1. Open a fresh session in an AI assistant (like ChatGPT, Claude, or Gemini).
2. Copy the entire prompt provided in the box below.
3. Paste it into the AI assistant and hit send.
4. Copy the Python code it generates, paste it into a new cell in this notebook, and run it

<br>

**👇 COPY THE PROMPT BELOW 👇**

Create a Python visualization showing the distributions of the following variables from the pandas DataFrame `df_raw`:

```python
SHOW = ["age", "Jitter(%)", "Shimmer", "NHR", "HNR"]
TARGET_NAME = "total_UPDRS"
```

### Requirements:

* Use `df_raw` as the source DataFrame.
* Create one histogram for each variable in `SHOW`, plus one for the target variable (`TARGET_NAME`). 
* Clearly identify the target variable in the plot (e.g., using a different color and title).
* Display all six distributions together in a clean, publication-quality figure.
* Use a multi-panel layout (exactly 2 rows × 3 columns).
* Use a consistent binning strategy and formatting across the feature distributions so they are easy to compare.
* Clearly label each subplot with the original variable name.
* Include the total number of rows (N) in the main figure title.
* Keep the visualization compact and suitable for inclusion in a scientific report or presentation.
* Hide unnecessary plot spines (top and right) and avoid excessive visual decoration.
* Use `matplotlib` for the visualization.
* Provide informative comments in the code for the key code lines.

Return complete, executable Python code that assumes `df_raw` already exists.

In [ ]:
# Copy the resulting generated code here and run the cell

Expected result should look similar to this one

In [ ]:
import matplotlib.pyplot as plt

# Variables definition
SHOW = ["age", "Jitter(%)", "Shimmer", "NHR", "HNR"]
TARGET_NAME = "total_UPDRS"

# Variables combined for iteration (5 features + 1 target = 6 panels)
variables_to_plot = SHOW + [TARGET_NAME]

# Create figure and a 2x3 grid of subplots
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 7))
axes = axes.flatten()

# Colors for clear visual separation
feature_color = "#4C72B0"  # Professional steel blue for features
target_color = "#C44E52"   # Muted red to highlight the target

# Plot distributions
for i, var in enumerate(variables_to_plot):
    ax = axes[i]
    
    # Identify if the current variable is the target
    is_target = (var == TARGET_NAME)
    color = target_color if is_target else feature_color
    title = f"{var} [TARGET]" if is_target else var
    
    # Plot histogram with consistent binning (30 bins) and dropna() for safety
    ax.hist(df_raw[var].dropna(), bins=30, color=color, edgecolor='white', linewidth=0.7)
    
    # Clean up labels and titles
    ax.set_title(title, fontsize=12, pad=10, fontweight='bold' if is_target else 'medium')
    ax.set_xlabel("Value", fontsize=10)
    ax.set_ylabel("Frequency", fontsize=10)
    
    # Hide unnecessary plot spines to avoid visual clutter
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add a subtle grid on the y-axis to help read frequency values
    ax.yaxis.grid(True, linestyle='--', alpha=0.5)
    
    # Ensure grid lines stay behind the histogram bars
    ax.set_axisbelow(True)

# Add an overall figure title that includes summary statistics (number of rows)
total_rows = len(df_raw)
fig.suptitle(f"Distributions of Selected Features and Target Variable (N = {total_rows:,} observations)", 
             fontsize=14, fontweight='bold')

# Automatically adjust subplots to fit into the figure area cleanly
plt.tight_layout()

# Display the visualization
plt.show()

---
## Visualisation Exercise 2: Variables relationships with a Scatter Matrix

Now that we have looked at the distributions of individual variables, we want to see how they relate to one another. 

Are any of the voice measures highly correlated? More importantly, does any single voice measure clearly predict our target?

**How to read a scatter matrix:**
* **Off-diagonal panels:** These are scatter plots comparing the row variable (y-axis) against the column variable (x-axis). A tight, diagonal cloud means the two variables move together (high correlation). A shapeless blob means they do not.
* **Diagonal panels:** These show the single-variable distribution (a histogram what we already did in the first part).

**What to look for once you generate the plot:**
Pay close attention to the **bottom row** (or the rightmost column), which will plot our target variable (`total_UPDRS`) against each voice measure. 
*Spoiler alert:* You will likely see **shapeless vertical/horizontal bands**. This visually demonstrates that there is no obvious, simple linear relationship between any *single* voice measure and the severity of the disease. It hints at why we need machine learning: combining multiple weak signals to make a strong prediction!


Writing a scatter matrix from scratch in `matplotlib` requires dozens of lines of code to handle grid placement, axes, ticks, and `for` loops. By using a specialized visualization library like `seaborn`, this can be done in just a few lines. Let's ask our AI assistant to write it for us.

**Instructions:**
1. Open your AI assistant session.
2. Copy the entire prompt in the box below.
3. Paste it into the AI assistant and hit send.
4. Copy the Python code it generates, paste it into a new cell in this notebook, and run it!

<br>

**👇 COPY THE PROMPT BELOW 👇**

> Create a Python visualization using `seaborn` to generate a scatter matrix (pairplot) from a pandas DataFrame named `df_raw`.
> 
> Variables to include:
> ```python
> features = ["Jitter(%)", "Shimmer", "NHR", "HNR", "PPE"]
> TARGET_NAME = "total_UPDRS"
> columns_to_plot = features + [TARGET_NAME]
> ```
> 
> ### Requirements:
> * Filter `df_raw` to only include `columns_to_plot`.
> * Use `seaborn.pairplot()` to create the scatter matrix.
> * For the diagonal subplots, plot histograms (`diag_kind='hist'`) with a unified color (e.g., steel blue).
> * For the off-diagonal scatter plots, handle data overlapping (overplotting) by setting a small marker size and lowering the opacity (e.g., `plot_kws={'alpha': 0.15, 's': 10, 'color': '#4C72B0'}`).
> * Apply a clean, professional Seaborn theme before plotting (e.g., `sns.set_theme(style="white")`).
> * Add an overarching title to the figure: "Scatter Matrix: Voice Measures vs Target (total_UPDRS)". 
> * Adjust the title spacing (`plt.subplots_adjust`) so it does not overlap with the top row of plots.
> * Return complete, executable Python code assuming `df_raw` already exists. Ensure necessary libraries (`pandas`, `seaborn`, `matplotlib.pyplot`) are imported.

In [ ]:
# Scatter matrix code
# Copy the resulting generated code here and run the cell 

---
This is an example of the expected resulting code : 

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assume df_raw is already defined in your environment
# df_raw = pd.read_csv('your_data.csv') 

# Define the features and target variable
features = ["Jitter(%)", "Shimmer", "NHR", "HNR", "PPE"]
target = "total_UPDRS"
columns_to_plot = features + [target]

# Filter the DataFrame to include only the specified columns
df_filtered = df_raw[columns_to_plot]

# Apply a clean, professional Seaborn theme
sns.set_theme(style="white")

# Define color for uniformity (Steel Blue)
steel_blue = "#4C72B0"

# Generate the scatter matrix (pairplot)
g = sns.pairplot(
    df_filtered,
    diag_kind='hist',
    plot_kws={'alpha': 0.15, 's': 10, 'color': steel_blue}, # Off-diagonal scatter plot settings
    diag_kws={'color': steel_blue}                          # Diagonal histogram settings
)

# Add an overarching title to the figure
g.fig.suptitle("Scatter Matrix: Voice Measures vs Target (total_UPDRS)")

# Adjust the title spacing so it doesn't overlap with the top row of subplots
plt.subplots_adjust(top=0.95)

# Display the plot
plt.show()

### What the raw data **doesn't** tell you

Everything above is what an AI (or a data scientist) would see if you simply printed `df.head()` and `df.dtypes`. But a pure table misses critical real-world context. Here is what is *not* captured in the raw data types:

1. **5,875 rows are not 5,875 people. They are 42 people.** Each patient contributed 101–168 recordings, spread over roughly 24 weekly sessions. This means the data is nested — *phonations inside sessions inside patients* — and rows belonging to the same patient are heavily correlated.
2. **`subject#` is an identifier, not a quantity.** Patient 17 is not "one more than" patient 16. It is stored as an integer (`int64`), so machine learning libraries will happily (and incorrectly) treat it as a numerical feature if you let them.


Point 1 is the most important because it defines our *scientific question*, which ultimately determines whether our analysis is right or wrong. 

The clinical promise of telemonitoring is: **"Hand this device to a patient we have never recorded before, and accurately estimate their disease severity."** 
To achieve this, our model must generalize **to new people**, not just memorize new recordings from people it has already seen.

---
## Visualisation Exercise 3 : Quantifying the Patient Effect

Let's confirm Point 1 numerically, because it drives everything we do next. We want to know: *How much of the variance in our target (`total_UPDRS`) is just driven by WHO the patient is, rather than how their voice changes over time?*

Instead of writing the complex Pandas `groupby` and variance calculations manually, let's have our AI assistant write a clean script for us.

**Instructions:**
1. Open your AI assistant session.
2. Copy the entire prompt in the box below.
3. Paste it into the AI assistant and hit send.
4. Copy the Python code it generates, paste it into a new cell in this notebook, and run it!

<br>

**👇 COPY THE PROMPT BELOW 👇**

> I have a pandas DataFrame named `df_raw`. 
> The dataset has a group identifier column called `subject#` and a continuous target column called `total_UPDRS`.
> 
> Please write Python code to analyze how much of the variance in the target is explained purely by the subject identity. 
> 
> ### Requirements:
> 1. Calculate and print the total number of rows and the number of unique subjects.
> I have a pandas DataFrame named `df_raw`. 
> The dataset has a group identifier column called `subject#` and a continuous target column called `total_UPDRS`.
> 
> Please write Python code to analyze how much of the variance in the target is explained purely by the subject identity. 
> 
> ### Requirements:
> 1. Calculate and print the total number of rows and the number of unique subjects.
> 2. Calculate and print the min, median, and max number of recordings (rows) per subject (use groupby and describe to do so).
> 3. Group the data by `subject#` and calculate the standard deviation (SD) of `total_UPDRS` in three ways, printing each:
>    - SD of the target *overall* (across the whole dataset).
>    - SD *between* subjects (the standard deviation of the subject means).
>    - SD *within* subjects (the mean of each subject's individual standard deviation).
> 4. Calculate the percentage of total variance explained by the subject differences. 
>    - Formula for between-subject sum of squares: `(group_sizes * (group_means - overall_mean) ** 2).sum()`
>    - Formula for total sum of squares: `((target_column - overall_mean) ** 2).sum()`
>    - Divide the between-subject variance by the total variance and print the result as a percentage (e.g., "XX%").
> 5. Create and display a histogram plot showing the distribution of the number of recordings (on the y-axis) per patient (patient id on x-axis). Ensure the plot includes an appropriate title and axis labels.
> 6. Print a concluding sentence formatting the percentage: "=> XX% of the variance in the outcome is driven by WHO the patient is, not how that patient changed over time."
> 
> Return only complete, executable Python code. Assume `df_raw`, `pandas`, and standard plotting libraries like `matplotlib.pyplot` are already loaded. Print statements should be nicely formatted and easy to read.

In [ ]:
# Paste the generate code here 

This is an example of the expected output

In [ ]:
# 1. Calculate and print the total number of rows and the number of unique subjects.
total_rows = len(df_raw)
unique_subjects = df_raw['subject#'].nunique()

print("--- Dataset Overview ---")
print(f"Total number of rows: {total_rows}")
print(f"Number of unique subjects: {unique_subjects}")
print("-" * 24 + "\n")

# 2. Calculate and print the min, median, and max number of recordings (rows) per subject.
recordings_per_subject = df_raw.groupby('subject#').size()
recordings_desc = recordings_per_subject.describe()

print("--- Recordings per Subject ---")
print(f"Min:    {recordings_desc['min']:.0f}")
print(f"Median: {recordings_desc['50%']:.0f}")
print(f"Max:    {recordings_desc['max']:.0f}")
print("-" * 30 + "\n")

# 3. Calculate the SD of total_UPDRS in three ways.
overall_sd = df_raw['total_UPDRS'].std()
group_means = df_raw.groupby('subject#')['total_UPDRS'].mean()
between_sd = group_means.std()
within_sd = df_raw.groupby('subject#')['total_UPDRS'].std().mean()

print("--- Standard Deviations ---")
print(f"Overall SD:         {overall_sd:.4f}")
print(f"Between subjects:   {between_sd:.4f}")
print(f"Within subjects:    {within_sd:.4f}")
print("-" * 27 + "\n")

# 4. Calculate the percentage of total variance explained by the subject differences.
overall_mean = df_raw['total_UPDRS'].mean()
group_sizes = recordings_per_subject
target_column = df_raw['total_UPDRS']

ss_between = (group_sizes * (group_means - overall_mean) ** 2).sum()
ss_total = ((target_column - overall_mean) ** 2).sum()

pct_variance = (ss_between / ss_total) * 100

print("--- Variance Analysis ---")
print(f"Variance explained by subject identity: {pct_variance:.2f}%")
print("-" * 25 + "\n")

# 5. Create and display a plot showing the number of recordings per patient.
plt.figure(figsize=(14, 6))
recordings_per_subject.plot(kind='bar', color='steelblue')
plt.title('Number of Recordings per Patient', fontsize=14)
plt.xlabel('Patient ID (subject#)', fontsize=12)
plt.ylabel('Number of Recordings', fontsize=12)
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.show()

# 6. Print the concluding sentence.
print(f"=> {pct_variance:.2f}% of the variance in the outcome is driven by WHO the patient is, not how that patient changed over time.")

---
## Round 1 : The lazy prompt

Now, let's try to code a machine learning model which predict the severity of the parkinson disease using a LLM and a prompt.

This is a general example of a vague prompt asking the LLM to guess what is expected as an outcome while providing minimal information and context.

Run the cell to print the message then copy the print output.

In [ ]:
PROMPT_1 = """My dataset is structured as pandas DataFrame called `df_raw`. It contains voice features to predict parkinson's disease.

Write Python code to predict the disease.
"""

print(PROMPT_1)

### Run prompt 1

- Copy the text (`PROMPT_1`) from the printed result above ☝️ 
- Paste into chat window then generate code.
- Paste the answer back into the empty cell below and run it

Then run the reference cell underneath, which reproduces a typical careful answer so that the notebook works end to end whatever your model did.

In [ ]:
# ⬇️⬇️⬇️  PASTE THE LLM ANSWER FOR PROMPT 1 HERE, THEN RUN THE CELL ⬇️⬇️⬇️


Example of the expected output

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# -------------------------------------------------------
# 1. Prepare data
# -------------------------------------------------------

# Change this if your target column has a different name
TARGET = "status"

# Remove rows where the target is missing
df = df_raw.dropna(subset=[TARGET]).copy()

# Separate features and target
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Keep only numeric voice features
X = X.select_dtypes(include=np.number)

print("Features:", X.shape[1])
print("Samples:", X.shape[0])
print("Classes:")
print(y.value_counts())

# -------------------------------------------------------
# 2. Train/test split
# -------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# -------------------------------------------------------
# 3. Build prediction pipeline
# -------------------------------------------------------

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

# -------------------------------------------------------
# 4. Train
# -------------------------------------------------------

model.fit(X_train, y_train)

# -------------------------------------------------------
# 5. Predict
# -------------------------------------------------------

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# -------------------------------------------------------
# 6. Evaluate
# -------------------------------------------------------

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

---
## What to expect and *why*

A language model doesn't know your dataset. It produces the **most probable continuation** of your words. So before reading the output, ask: *what did my prompt actually say?*


**Based on what we provided, the model might infer (correctly or not):**

- *"predict the disease"* → binary classification (disease vs. healthy), even if the target is a continuous severity score, making this a regression problem.
- *"voice features"* + *"parkinson"* → almost certainly a Parkinson's detection pipeline, pulled from similar examples in its training data.
- No column names given → the code will provide a filler for the target column name and treat **everything else** as a feature, including patient IDs, dates, or session numbers.
- No study design given → a naive random train/test split, even though clinical data typically has multiple recordings per patient. This leaks identity, not disease.


**The core issue:** the model confidently answers the question *your words implied*, not the one *you meant*. If every patient in your dataset already has the disease, a diagnosis classifier is meaningless, but the model can't know that. 

#### The confident tone isn't an evidence it knows about your subject, it's just the most probable output.

### A **loud** failure

The code never ran. You lost time, but you know unambiguously that something is wrong.


The model did nothing irrational. It received an under-specified request and returned the highest-probability Python code completion given the informations provided in the prompt and nothing else. 

**This is the least harmful failure mode.** An exception is the the best case scenario here: immediate, unambiguous, and acknowledgeable.

Round 1 failed loudly because the model guessed on element it did not know about. So fix that: give it more context. The guessing will not stop. It will just stop crashing.




---

## Round 2 : Pushing the prompt a little further

Round 1 failed because the model guessed what it had to do because we never gave it clear guidance. To fix it we can try to **give it the dataset structure and provide instructions on what we expect.** 

This time the code will run. It will run cleanly. It will drop the identifier, cross-validate itself, print three metrics, draw a residual plot.

**However, result will be flawed.**

That combination : correct syntax, sound-looking method, plausible result wrong conclusion is why we ALWAYS have to be careful when using LLM for coding. There will be times where your results will look just fine but in the end, you will end up with fallacious output.  

First, generate the schema you are going to paste.

In [ ]:
schema = "\n".join(f"  {c:<14} {t}" for c, t in df_raw.dtypes.items() if c in VOICE)

PROMPT_2 = f"""I have a pandas DataFrame `df_raw` with 21 columns holding clinical scores for Parkinson's telemonitoring. There are no missing values.

Columns and dtypes:
{schema}

Build a machine learning model that predicts `total_UPDRS` as a regression task.
Follow these steps exactly:
1. Only use the columns in the Python list `VOICE` as the features. Do not infer, rename, add, or remove feature columns.
2. Define:
   - `X = df_raw[VOICE]`
   - `y = df_raw["total_UPDRS"]`
3. Do not use `subject#` or `test_time` as features.
4. Perform a standard 80/20 random train/test split.
5. Train an appropriate regression model.
6. Evaluate the model on the test set using:
   - MAE
   - RMSE
   - R²
7. Plot predicted vs. actual `total_UPDRS` values.
8. Provide complete, executable Python code.
"""


print(PROMPT_2)

### Run Prompt 2 :
- Copy the text from the printed result above ☝️ 
- Paste into chat window then generate code.
- Paste the answer back into the empty cell below and run it

Then run the reference cell underneath, which reproduces a typical careful answer so that the notebook works end to end whatever your model did.

In [ ]:
# ⬇️⬇️⬇️  PASTE THE LLM ANSWER FOR PROMPT 2 HERE, THEN RUN THE CELL ⬇️⬇️⬇️

Reference result :

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ---------------------------------------------------------------------
# 1. Define the voice features
# ---------------------------------------------------------------------

VOICE = [
    "Jitter(%)",
    "Jitter(Abs)",
    "Jitter:RAP",
    "Jitter:PPQ5",
    "Jitter:DDP",
    "Shimmer",
    "Shimmer(dB)",
    "Shimmer:APQ3",
    "Shimmer:APQ5",
    "Shimmer:APQ11",
    "Shimmer:DDA",
    "NHR",
    "HNR",
    "RPDE",
    "DFA",
    "PPE",
]


# ---------------------------------------------------------------------
# 2. Define X and y exactly as requested
# ---------------------------------------------------------------------

X = df_raw[VOICE]
y = df_raw["total_UPDRS"]


# ---------------------------------------------------------------------
# 3. Train/test split: standard 80/20 random split
# ---------------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)


# ---------------------------------------------------------------------
# 4. Train a regression model
# ---------------------------------------------------------------------

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)


# ---------------------------------------------------------------------
# 5. Predict on the test set
# ---------------------------------------------------------------------

y_pred = model.predict(X_test)


# ---------------------------------------------------------------------
# 6. Evaluate
# ---------------------------------------------------------------------

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Test-set performance")
print("-" * 30)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


# ---------------------------------------------------------------------
# 7. Plot predicted vs. actual values
# ---------------------------------------------------------------------

plt.figure(figsize=(7, 7))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.6,
)

# Perfect-prediction reference line
min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--",
    linewidth=2,
)

plt.xlabel("Actual total_UPDRS")
plt.ylabel("Predicted total_UPDRS")
plt.title("Predicted vs. Actual total_UPDRS")

plt.tight_layout()
plt.show()

---
## What to expect and *why*

The generated code will run perfectly. It will drop the identifier, split the data, train the model, and print metrics showing moderate but promising performance (e.g., R² ≈ 0.35). 


The problem isn't in the code syntax; it is in the mismatch between standard ML recipes and specific field knowledge (scientist job!). By strictly following the prompt, the model commits a massive **data leakage** error. 

Here is exactly what goes wrong:

*   **The Missing Context:** The dataset contains roughly 140 voice recordings *per patient*. The prompt describes the columns, but doesn't explain this repeated-measures structure. 
*   **The Silent Failure:** By dropping `subject#` and applying a standard `train_test_split`, the code randomly shuffles the rows, assuming they are independent. They are not.
*   **The Leakage:** Every patient's ~140 recordings are scattered across both sides of the split. The model is trained on approximately 80% of Patient A's voice recordings and tested on the remaining 20%. 

**Why is this dangerous?** : 
Because the model isn't learning a generalizable biological link between vocal features and Parkinson's severity. 

It is simply learning to recognize each patient unique vocal profile and reciting their known `total_UPDRS` score. If you try this model on a brand new patient, it would fail completely.

## **It looks good but it is not! It is a silent failure !**
That is what makes it dangerous: **there is nothing problematic at first sight.** The suggested code returns believable, modest metrics (MAE ≈ 6.6, R² ≈ 0.37) with residual plots. 


`Prompt 1` gave you an exception. 
`Prompt 2` gives you a **result**: syntactically clean, methodologically conventional and numerically plausible. There is no traceback error to read, no line of code that is wrong and everything does precisely what it says.



---
## Round 3 : A specific prompt

## The anatomy of a good prompt

A prompt for research code is a list of **specification**. 

You role is to define each and every constrain required to perform the task.

Here is an example: 

| Block | What it answers | What it would have fixed here |
|---|---|---|
| **1. Role & environment** | Who is writing, which libraries, what deliverable | Stops imports you don't have; sets the standard of rigour |
| **2. Data contract** | Exact columns, dtypes, **units**, ranges, encodings, sentinels | Kills the `status` hallucination; declares `sex` 0/1 and negative `test_time` |
| **3. Domain constraints** | What the numbers *mean*, which are IDs, how the outcome was measured | Kills `subject#`-as-feature and the interpolation leak |
| **4. Design & unit of generalisation** | Nesting, repeated measures, **what the model must generalise to** | Kills subject leakage — the error that broke Round 2 |
| **5. Output specification** | Exact metrics, baselines, figure layout, labels | Gets you a number you can compare to something |
| **6. Validation & reproducibility** | Seeds, assertions, printed counts, docstrings | Lets you *prove* the result six months later |

Two things are worth noticing about that table.


**Almost none of these blocks require you to be really good at Python programming.** They require you to know your field of expertise. 

## Step 1: Building Your Specifications List

When working with Large Language Models in scientific coding and data analysis, the biggest mistake beginners make is asking the model to "analyze this data" without any guardrails. If you don't give the LLM specific rules, it will guess and might often guess wrong.

To prevent this, your first step is to create a **Specification List**. Think of this as a major guidelines for your project. A good specification list covers elements such as:

1. **Constraints you know:** What columns should the model *never* use? What are the limitations of the data? (e.g., "Patient ID is a label, not a number to do math on.")
2. **The evaluation you want to perform:** How should the model be tested? (e.g., "Test on patients the model has never seen before.")
3. **The algorithms to apply:** What models should be used? (e.g., "Use a Random Forest, and compare it to a baseline Dummy model.")
4. **The results you expect:** What should the final output look like? (e.g., "Give me a single Python cell, a 2-panel plot, and a 4-sentence plain-language conclusion.")

Along with these, always provide **context**. Tell the model *what the numbers actually mean* and *how they were collected*. Domain knowledge lives in your head, not in the dataset. By explaining the context, the model transforms from a blind calculator into a smart assistant.

### Agentic Advantage 🤖 (Claude Code, , Codex, Cursor, ...)
As AI coding tools evolve into "Agentic" workflows, building these specifications becomes much easier. Instead of you having to manually type out every detail about your data and your code base (like column names, exact row counts, data types, availables functions and existing code), agentic tools can:
* Explore your codebase and data frames automatically.
* Run code in the background to learn about the context.
* Ask you clarifying questions before they start writing code.

However, **agents still need your domain knowledge.** An AI can see that a column is named `test_time`, but only *you* know that using it as a predictor will ruin the experiment because of how the data was collected. 

### "Meta-Prompting" Strategy
We are going to use a technique called **Meta-Prompting**. 
Instead of trying to write the "perfect, highly detailed prompt" yourself, you will:
1. Write a simple, plain-English bulleted list of your specifications.
2. Give that list to an LLM and ask *it* to write the perfect, highly structured prompt.
3. Pass that newly generated prompt into a fresh LLM session to actually write your code.

***

## Part 2: Specification List

In [ ]:
DF_NAME = "df_raw"
TARGET  = meta["target"]            # e.g., "total_UPDRS"
GROUP   = meta["group_col"]         # e.g., "subject#"
TIME    = meta["sequence_col"]      # e.g., "test_time"
VOICE   = meta["voice_cols"]        # List of the acoustic measure column names
N_FOLDS = 5

n_patients = df_raw[GROUP].nunique()
n_voice = len(VOICE)
voice_cols_str = ", ".join(VOICE)

META_PROMPT = f"""
I am a data scientist. I want you to act as an expert prompt engineer. 
I have written a list of specifications for a machine learning analysis I want to run on my dataset. 
Please take my bullet points below and turn them into a detailed, strictly structured prompt 
that I can feed into a new AI to generate the exact Python code I need. 

Here are my specifications:

*   **The Goal:** Predict Parkinson's disease severity (`{TARGET}`) based purely on voice acoustic measures. This is a regression problem.
*   **The Data:** 
    *   The dataframe is already loaded in memory and called `{DF_NAME}`.
    *   The target column is `{TARGET}`.
    *   The group column is `{GROUP}` (Patient ID). There are {n_patients} unique patients in this cohort.
    *   The voice columns (predictors) are exactly these {n_voice} acoustic variables: {voice_cols_str}.
*   **Domain Constraints (Crucial):**
    *   Everyone in the dataset has the disease; there are no healthy controls.
    *   Do NOT use `{GROUP}`, `age`, or `sex` as predictors. Age and sex are constant per patient, so the model might just use them to memorize patients.
    *   Solely use the acoustic variables voice columns as predictors : {voice_cols_str}.
    *   Drop `motor_UPDRS` entirely. It's just a different scoring of the same exam; predicting one from the other is cheating.
    *   Drop `{TIME}`. The target was interpolated over time, so using time as a predictor will just reverse-engineer the math, not biology.
    *   The voice predictors are on different scales so any scale-sensitive model must be wrapped in a pipeline with StandardScaler.
*   **Evaluation (Design):**
    *   Patients have multiple recordings (repeated measures). 
    *   The clinical question is: "Can we predict severity for a *new* patient?" 
    *   Therefore, the model must be validated using `GroupKFold` ({N_FOLDS} folds) grouped by `{GROUP}`. No patient can be in both train and test splits! Never use `train_test_split`.
*   **Algorithms:**
    *   Use a `RandomForestRegressor` (100 trees).
    *   Compare it against a `DummyRegressor` (strategy="mean") so we have a baseline floor to prove the model actually learned something.
*   **Expected Output:**
    *   Return ONE complete, runnable Python cell using pandas, scikit-learn, and matplotlib.
    *   Calculate Mean Absolute Error (MAE) and R^2 for both models and report std and mean of all folds.
    *   Include a figure with two panels: Left showing per-fold MAE for both models, Right showing out-of-fold predicted vs true values with an identity line.
*   **Validation:** 
    *   Include `assert` statements in the code to guarantee no patient data leaked across folds, and that forbidden columns weren't used as predictors.
    *   Fix every random_state to {RANDOM_SEED}.
"""

print(META_PROMPT)

---
### This is the kind of prompt your model should generated based on you meta-prompt (and list specification)

## Example of a generated prompt from the meta-prompt


> **Role:** 
> Act as an expert Machine Learning Engineer and Python Data Scientist. Your task is to write strict, production-ready, and highly validated Python code for a specific medical machine learning regression problem.
> 
> **Context & Goal:** 
> Predict Parkinson's disease severity (target variable: `total_UPDRS`) based purely on voice acoustic measures. 
> The dataset is already loaded in memory as a pandas DataFrame named `df_raw`.
> 
> **1. Data Definition & Strict Preprocessing Rules:**
> *   **Target Column:** `total_UPDRS`
> *   **Group Column:** `subject#` (represents 42 unique Patient IDs).
> *   **Allowed Predictors (Features):** You must strictly construct your feature matrix `X` using *only* the following 16 acoustic variables: `Jitter(%)`, `Jitter(Abs)`, `Jitter:RAP`, `Jitter:PPQ5`, `Jitter:DDP`, `Shimmer`, `Shimmer(dB)`, `Shimmer:APQ3`, `Shimmer:APQ5`, `Shimmer:APQ11`, `Shimmer:DDA`, `NHR`, `HNR`, `RPDE`, `DFA`, `PPE`.
> *   **Forbidden Columns (Crucial Domain Constraints):** 
>     *   Do NOT use `subject#`, `age`, or `sex` as predictors. (Age and sex are constant per patient; the model will memorize patients instead of learning voice pathology).
>     *   Do NOT use `motor_UPDRS` (This is a subset of the target score; using it is data leakage/cheating).
>     *   Do NOT use `test_time` (The target score was mathematically interpolated over time; using time reverses the math rather than learning biology).
> *   **Scaling:** Build a scikit-learn `Pipeline` that applies `StandardScaler` to the features prior to modeling to account for differing acoustic scales.
> 
> **2. Cross-Validation & Evaluation Strategy:**
> *   **Algorithm:** Compare a `RandomForestRegressor(n_estimators=100, random_state=42)` against a baseline `DummyRegressor(strategy="mean")`.
> *   **Validation Scheme:** Because patients have repeated measures (multiple recordings per `subject#`), we are simulating predicting severity for a *new* patient. You MUST use `GroupKFold(n_splits=5)` grouped by `subject#`. 
> *   **Strict Constraint:** Never use `train_test_split` or standard `KFold`. A patient cannot exist in both the train and test sets simultaneously.
> *   **Metrics:** Calculate and collect out-of-fold Mean Absolute Error (MAE) and $R^2$ for both models. 
> 
> **3. Safety & Validation (Assert Statements):**
> Before generating the metrics, your code must include explicit `assert` statements that dynamically check the current fold during the CV loop to guarantee data integrity:
> 1.  Assert that the intersection of `subject#` arrays in the training fold and testing fold is completely empty (Guarantees no patient leakage).
> 2.  Assert that forbidden columns (`subject#`, `age`, `sex`, `motor_UPDRS`, `test_time`) are completely absent from the feature matrix `X`.
> 
> **4. Output Requirements:**
> *   **Code Format:** Output exactly ONE complete, self-contained, and runnable Python code cell. Do not fragment the code. Assume `pandas`, `sklearn`, `numpy`, and `matplotlib.pyplot` need to be imported.
> *   **Reproducibility:** Fix `random_state=42` everywhere applicable (RandomForest, cross-validation if applicable, etc.).
> *   **Visualizations:** Use `matplotlib` to generate a single figure with two subplots (1 row, 2 columns):
>     *   **Left Panel:** A bar chart or grouped bar chart comparing the per-fold MAE of the RandomForest against the DummyRegressor to visually prove the model learned something beyond the baseline.
>     *   **Right Panel:** A scatter plot of Out-of-Fold (OOF) Predicted values vs. True `total_UPDRS` values for the RandomForest model. Include a dashed diagonal identity line (y=x) to represent perfect predictions.
> *   **Console Output:** Print the aggregated cross-validated MAE and $R^2$ for both the RandomForest and the DummyRegressor clearly at the end of the script.
> 
> **Do not provide filler text. Output the requested Python code and brief comments explaining the steps.**

### Run Prompt 3 :
- **Copy the prompt your model generated** from the meta prompt OR use the one above ☝️ 
- Paste into chat window then generate code.
- Paste the answer back into the empty cell below and run it

Then run the reference cell underneath, which reproduces a typical careful answer so that the notebook works end to end whatever your model did.

In [ ]:
# ⬇️⬇️⬇️  PASTE THE LLM ANSWER FOR PROMPT 3 HERE, THEN RUN THE CELL ⬇️⬇️⬇️

Expected result :

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score


# ---------------------------------------------------------------------
# 1. Define target, grouping variable, and strictly allowed features
# ---------------------------------------------------------------------

TARGET = "total_UPDRS"
GROUP = "subject#"

FEATURES = [
    "Jitter(%)",
    "Jitter(Abs)",
    "Jitter:RAP",
    "Jitter:PPQ5",
    "Jitter:DDP",
    "Shimmer",
    "Shimmer(dB)",
    "Shimmer:APQ3",
    "Shimmer:APQ5",
    "Shimmer:APQ11",
    "Shimmer:DDA",
    "NHR",
    "HNR",
    "RPDE",
    "DFA",
    "PPE",
]

FORBIDDEN_COLUMNS = {
    "subject#",
    "age",
    "sex",
    "motor_UPDRS",
    "test_time",
}

# Validate that all required columns exist.
required_columns = set(FEATURES) | {TARGET, GROUP}
missing_columns = required_columns - set(df_raw.columns)
assert not missing_columns, (
    f"Missing required columns: {sorted(missing_columns)}"
)

# Construct X strictly from the 16 acoustic features.
X = df_raw[FEATURES].copy()
y = df_raw[TARGET].copy()
groups = df_raw[GROUP].copy()

# Explicitly guarantee that forbidden variables cannot enter X.
assert FORBIDDEN_COLUMNS.isdisjoint(X.columns), (
    f"Forbidden columns found in X: "
    f"{sorted(FORBIDDEN_COLUMNS.intersection(X.columns))}"
)

assert list(X.columns) == FEATURES, "X does not contain exactly the allowed features."
assert len(X) == len(y) == len(groups), "X, y, and groups have inconsistent lengths."


# ---------------------------------------------------------------------
# 2. Define leakage-safe models
#
# Scaling is fitted independently inside each training fold because
# StandardScaler is part of the Pipeline.
# ---------------------------------------------------------------------

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
    )),
])

dummy_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", DummyRegressor(strategy="mean")),
])


# ---------------------------------------------------------------------
# 3. Grouped 5-fold cross-validation
#
# Every patient is entirely contained in either the training or testing
# portion of each fold.
# ---------------------------------------------------------------------

cv = GroupKFold(n_splits=5)

rf_mae_folds = []
rf_r2_folds = []
dummy_mae_folds = []
dummy_r2_folds = []

# Store OOF predictions for the Random Forest.
oof_true = np.empty(len(y), dtype=float)
oof_rf_pred = np.empty(len(y), dtype=float)
oof_filled = np.zeros(len(y), dtype=bool)

for fold, (train_idx, test_idx) in enumerate(
    cv.split(X, y, groups=groups),
    start=1,
):
    # -------------------------------------------------------------
    # Explicit fold-level leakage checks
    # -------------------------------------------------------------

    train_subjects = groups.iloc[train_idx].to_numpy()
    test_subjects = groups.iloc[test_idx].to_numpy()

    # 1. No patient may occur in both training and testing folds.
    assert len(np.intersect1d(train_subjects, test_subjects)) == 0, (
        f"Patient leakage detected in fold {fold}."
    )

    # 2. Forbidden predictors must remain absent from X.
    assert FORBIDDEN_COLUMNS.isdisjoint(X.columns), (
        f"Forbidden columns detected in X during fold {fold}: "
        f"{sorted(FORBIDDEN_COLUMNS.intersection(X.columns))}"
    )

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    # Fit only on the training patients.
    rf_pipeline.fit(X_train, y_train)
    dummy_pipeline.fit(X_train, y_train)

    # Predict only on previously unseen patients.
    rf_pred = rf_pipeline.predict(X_test)
    dummy_pred = dummy_pipeline.predict(X_test)

    # Fold-level metrics.
    rf_mae_folds.append(mean_absolute_error(y_test, rf_pred))
    rf_r2_folds.append(r2_score(y_test, rf_pred))

    dummy_mae_folds.append(mean_absolute_error(y_test, dummy_pred))
    dummy_r2_folds.append(r2_score(y_test, dummy_pred))

    # Save Random Forest out-of-fold predictions.
    oof_true[test_idx] = y_test.to_numpy()
    oof_rf_pred[test_idx] = rf_pred
    oof_filled[test_idx] = True


# Every observation must have exactly one OOF prediction.
assert np.all(oof_filled), "Some observations do not have an OOF prediction."


# ---------------------------------------------------------------------
# 4. Aggregate OOF metrics
#
# The reported MAE and R² are calculated from all OOF predictions,
# rather than averaging the five fold scores. This gives each
# observation equal weight.
# ---------------------------------------------------------------------

rf_oof_mae = mean_absolute_error(oof_true, oof_rf_pred)
rf_oof_r2 = r2_score(oof_true, oof_rf_pred)

# Reconstruct DummyRegressor OOF predictions so its aggregate metrics
# can be calculated in exactly the same way.
oof_dummy_pred = np.empty(len(y), dtype=float)

for train_idx, test_idx in cv.split(X, y, groups=groups):
    dummy_pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
    oof_dummy_pred[test_idx] = dummy_pipeline.predict(X.iloc[test_idx])

dummy_oof_mae = mean_absolute_error(oof_true, oof_dummy_pred)
dummy_oof_r2 = r2_score(oof_true, oof_dummy_pred)


# ---------------------------------------------------------------------
# 5. Visualization
# ---------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: per-fold MAE comparison.
fold_numbers = np.arange(1, 6)
width = 0.35

axes[0].bar(
    fold_numbers - width / 2,
    rf_mae_folds,
    width,
    label="Random Forest",
)
axes[0].bar(
    fold_numbers + width / 2,
    dummy_mae_folds,
    width,
    label="Dummy Regressor",
)

axes[0].set_xlabel("CV Fold")
axes[0].set_ylabel("MAE")
axes[0].set_title("Per-Fold MAE")
axes[0].set_xticks(fold_numbers)
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)


# Right: Random Forest OOF predictions vs true values.
axes[1].scatter(
    oof_true,
    oof_rf_pred,
    alpha=0.5,
)

plot_min = min(oof_true.min(), oof_rf_pred.min())
plot_max = max(oof_true.max(), oof_rf_pred.max())

axes[1].plot(
    [plot_min, plot_max],
    [plot_min, plot_max],
    linestyle="--",
    label="Perfect prediction (y=x)",
)

axes[1].set_xlabel("True total_UPDRS")
axes[1].set_ylabel("OOF Predicted total_UPDRS")
axes[1].set_title("Random Forest: Out-of-Fold Predictions")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------
# 6. Console summary
# ---------------------------------------------------------------------

print("\n" + "=" * 65)
print("GROUPED 5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 65)

print("\nRandom Forest:")
print(f"  OOF MAE: {rf_oof_mae:.4f}")
print(f"  OOF R²:  {rf_oof_r2:.4f}")
print(f"  Mean fold MAE: {np.mean(rf_mae_folds):.4f}")
print(f"  Mean fold R²:  {np.mean(rf_r2_folds):.4f}")

print("\nDummy Regressor:")
print(f"  OOF MAE: {dummy_oof_mae:.4f}")
print(f"  OOF R²:  {dummy_oof_r2:.4f}")
print(f"  Mean fold MAE: {np.mean(dummy_mae_folds):.4f}")
print(f"  Mean fold R²:  {np.mean(dummy_r2_folds):.4f}")

print("\n" + "=" * 65)

## What to expect and *why*

**Expect a much worse number. That is the point!**

Concretely, expect:

- `GroupKFold(n_splits=5)` with `groups=df["subject#"]`, and no `train_test_split` anywhere;
- exactly 16 predictors, `subject#` / `age` / `sex` / `test_time` / `motor_UPDRS` all gone;
- a random forest whose **MAE is around 9–10 UPDRS points**;
- a dummy baseline whose **MAE is around 8.9 UPDRS points**;
- an **R² near zero or negative** even around −0.4 for the forest;

**Why the honest answer is so bad.** 

Under a grouped split the model is tested on patients whose UPDRS level it has never seen. The between-patient variation it exploited in Round 2 is now precisely what it must predict and it fails to do so. The forest ends up doing *worse than predicting the mean*, which is what a negative R² means: it uses voice features to move its predictions away from the group average in the wrong direction.

Round 2 said `R² ~ 0.35`. Round 3 says `R² ≈ −0.4`. **Same data.** The 0.35 was measuring how well a model can recognise a voice it has already heard.

**This is a real result, not a broken one.** "16 dysphonia measures do not generalise to unseen patients in a 42-person cohort" is a finding even if not a nice one!

Same data. Same model class. Mostly the same method.

The only thing that changed between those two panels is **how much of your problem knowledge you provided to the model.**

---

## What to take away

- **The model follows your instructions, so it need you to provide enough context to perform at its best** : It cannot see your file, your cohort, or your recruitment protocol. Every fact you omit is silently replaced by the most common pattern in its training data leading to wrongful outputs.

- **A good prompt is a methods section written in advance.** If you cannot write the prompt, you do not yet understand what you expect the model to do. No model will understand it for you. Conversely, once you *can* write it, the model becomes genuinely excellent at the part it is good at: turning a clear specification into correct code, fast.

- **You remain the author.** You are responsible of validating the code and the results. Read every generated code line, and ALWAYS validate that the results are aligned with what we expect.
